# Chunking

**Goal:** Load the processed documentation pages from the previous notebook, split them into retrieval-ready `Chunk` objects, and inspect the results before embedding.

The `ChunkSplitter` strategy (documented in `src/chunking/chunker.py`):
1. **Primary split** on the `<!-- section/api: … -->` markers the converter embedded at every structural boundary.
2. **Merge forward** — stub segments < `min_chars` are prepended to the next segment.
3. **Sub-split** — segments > `max_chars` are sliced at paragraph boundaries with a configurable overlap tail.

## 1. Environment Setup

In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

DATA_DIR = "data"

PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", sys.path[0])

from src import data_acquisition as da
from src import chunking as ck

Project root: /home/dmitry/Projects/DataScience/rag-techdoc-assistant


In [2]:
import json
import logging
from collections import Counter

logging.basicConfig(
    level=logging.WARNING,
    format="%(asctime)s  %(name)-20s  %(levelname)-8s  %(message)s",
    datefmt="%H:%M:%S",
    force=True,
)
log = logging.getLogger("notebook")

## 2. Load Saved Pages

In [3]:
OUTPUT_DIR = PROJECT_ROOT / DATA_DIR / "pytorch_docs"

pages = da.pipeline.load_pages_from_disk(OUTPUT_DIR)

print(f"Loaded {len(pages)} pages from {OUTPUT_DIR}")

Loaded 2750 pages from /home/dmitry/Projects/DataScience/rag-techdoc-assistant/data/pytorch_docs


## 3. Configure the Splitter

In [4]:
splitter = ck.ChunkSplitter(
    max_chars=1500,
    min_chars=120,
    overlap_chars=200,
)

print("ChunkSplitter config:")
print(f"  max_chars     = {splitter.max_chars}")
print(f"  min_chars     = {splitter.min_chars}")
print(f"  overlap_chars = {splitter.overlap_chars}")

ChunkSplitter config:
  max_chars     = 1500
  min_chars     = 120
  overlap_chars = 200


## 4. Single-page Dry Run

In [5]:
sample = pages[51]
sample_chunks = splitter.split(sample)

print(f">> Page   : {sample.title}\n")
print(f">> Symbols: {sample.symbols}\n")
print(f">> Chunks : {len(sample_chunks)}")
print()

for i, chunk in enumerate(sample_chunks):
    cont = " (continuation)" if chunk.is_continuation else ""
    print(f"  [{i}] kind={chunk.kind:<12} chars={chunk.char_count:>5}  anchor={chunk.anchor}{cont}")

>> Page   : torch.func

>> Symbols: []

>> Chunks : 4

  [0] kind=heading      chars=  664  anchor=#torch-func
  [1] kind=heading      chars=  649  anchor=#what-are-composable-function-transforms
  [2] kind=heading      chars=  602  anchor=#why-composable-function-transforms
  [3] kind=heading      chars=  465  anchor=#read-more


In [6]:
# Inspect a specific chunk in detail
idx = 3
c = sample_chunks[idx]
print(f">> chunk_id     : {c.chunk_id}")
print(f">> citation_url : {c.citation_url}")
print(f">> kind         : {c.kind}")
print(f">> symbol       : {c.symbol or '—'}")
print(f">> keywords     : {c.keywords}")
print()
print("─" * 60)
print(c.text)

>> chunk_id     : read_more__0__31fff3
>> citation_url : https://docs.pytorch.org/docs/stable/func.html#read-more
>> kind         : heading
>> symbol       : —
>> keywords     : ['torch', 'func', 'What', 'are', 'composable', 'function', 'transforms', 'Why', 'Read', 'More']

────────────────────────────────────────────────────────────
## Read More

- torch.func Whirlwind Tour
  - What is torch.func?
  - Why composable function transforms?
  - What are the transforms?
- torch.func API Reference
  - Function Transforms
  - Utilities for working with torch.nn.Modules
  - Debug utilities
- UX Limitations
  - General limitations
  - torch.autograd APIs
  - vmap limitations
  - Randomness
- Migrating from functorch to torch.func
  - function transforms
  - NN module utilities
  - functorch.compile


## 5. Split All Pages

In [7]:
from src.chunking import split_incremental

CHUNKS_PATH = OUTPUT_DIR / "_chunks.jsonl"

all_chunks = split_incremental(
    pages=pages,
    chunks_path=CHUNKS_PATH,
    splitter=splitter,
)

print(f"Total chunks : {len(all_chunks):,}")
print(f"Avg per page : {len(all_chunks) / max(len(pages), 1):.1f}")

Total chunks : 8,358
Avg per page : 3.0


## 6. Quality Checks

In [8]:
import statistics

char_counts = [c.char_count for c in all_chunks]
kind_counts = Counter(c.kind for c in all_chunks)

print("Chunk size distribution:")
print(f"  min    : {min(char_counts):,} chars")
print(f"  median : {statistics.median(char_counts):,.0f} chars")
print(f"  mean   : {statistics.mean(char_counts):,.0f} chars")
print(f"  p95    : {sorted(char_counts)[int(len(char_counts)*0.95)]:,} chars")
print(f"  max    : {max(char_counts):,} chars")
print()

Chunk size distribution:
  min    : 5 chars
  median : 496 chars
  mean   : 809 chars
  p95    : 2,327 chars
  max    : 47,275 chars



In [9]:
# Verify every chunk has a valid citation URL and non-empty text
problems = [
    c for c in all_chunks
    if not c.text.strip() or not c.citation_url
]

if problems:
    print(f"⚠  {len(problems)} chunks with missing text or citation URL:")
    for p in problems[:5]:
        print(f"   {p.chunk_id}")
else:
    print(f"All {len(all_chunks):,} chunks have non-empty text and citation URLs.")

All 8,358 chunks have non-empty text and citation URLs.


## 7. Save Chunks to Disk

In [10]:
CHUNKS_PATH = OUTPUT_DIR / "_chunks.jsonl"

with CHUNKS_PATH.open("w", encoding="utf-8") as f:
    for chunk in all_chunks:
        f.write(json.dumps(chunk.to_dict(), ensure_ascii=False) + "\n")

size_mb = CHUNKS_PATH.stat().st_size / 1e6
print(f"Saved {len(all_chunks):,} chunks: {CHUNKS_PATH}  ({size_mb:.1f} MB)")

Saved 8,358 chunks: /home/dmitry/Projects/DataScience/rag-techdoc-assistant/data/pytorch_docs/_chunks.jsonl  (17.1 MB)
